In [2]:
import numpy as np
import pandas
import sklearn 
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from mlxtend.plotting import plot_decision_regions
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, make_scorer
from sklearn import datasets
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [3]:
#import the data
data = pandas.read_csv('./Project.csv', delimiter=',')

In [4]:
#print the data
data

,user_id,day_type,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,burnout_score,burnout_risk
0,1,Weekday,9.59,11.86,4,2,0,7.55,91.2,19.17,Low
1,1,Weekend,7.38,10.33,4,1,0,6.69,82.0,29.70,Low
2,1,Weekend,6.31,8.92,1,2,0,8.87,80.6,32.93,Low
3,1,Weekday,8.34,10.70,4,1,1,8.13,70.0,45.47,Low
4,1,Weekend,6.97,9.83,1,2,0,5.85,67.1,51.61,Low
...,...,...,...,...,...,...,...,...,...,...,...
1795,180,Weekend,6.33,8.16,0,4,0,5.59,73.5,31.91,Low
1796,180,Weekend,4.70,7.88,0,4,0,6.69,89.8,26.30,Low
1797,180,Weekend,3.92,6.39,2,1,0,6.77,74.6,34.07,Low
1798,180,Weekday,8.93,11.11,2,5,0,8.28,74.6,38.14,Low


In [5]:
#set variables
hours_worked = data["work_hours"]
screen_time = data["screen_time_hours"]
meetings_count = data["meetings_count"]
breaks_taken = data["breaks_taken"]
after_hours_work = data["after_hours_work"]
sleep_hours = data["sleep_hours"]
task_completion = data["task_completion_rate"]
burnout_score = data["burnout_score"]
burnout_risk = data["burnout_risk"]

In [6]:
x = data.drop(columns = ['user_id','burnout_score'])
y = data['burnout_score']

In [7]:
x_nominal = pandas.get_dummies(x)

In [8]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=500)
xn_train, xn_test, yn_train, yn_test = train_test_split(x_nominal, y, test_size=500)

labels = ['model'] + x_nominal.columns.to_list()
labels
spearman_table = pandas.DataFrame(columns = labels)
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium


## Linear

### Linear Regression

In [9]:
linear = LinearRegression()

In [ ]:
linear.fit(xn_train, yn_train)
y_predicted_lr = linear.predict(xn_test)
mean_squared_error(yn_test, y_predicted_lr)

print("Mean Squared Error:", mean_squared_error(yn_test, y_predicted_lr))

# feature importance ranking
linear_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': np.abs(linear.coef_)}).sort_values(by='importance', ascending=False)

linear_importance

Mean Squared Error: 30.014491818307953


,feature,importance
9,burnout_risk_High,21.769745
10,burnout_risk_Low,15.018296
11,burnout_risk_Medium,6.751449
6,task_completion_rate,1.338784
4,after_hours_work,0.533264
0,work_hours,0.236636
1,screen_time_hours,0.235541
5,sleep_hours,0.225917
3,breaks_taken,0.065551
2,meetings_count,0.015837


In [12]:
y_predicted_lr = pandas.DataFrame(y_predicted_lr)

In [13]:
vals = {}
vals['model'] = 'linear regression'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_lr, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

C:\Users\adria\AppData\Local\Temp\ipykernel_6264\426421780.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])


### Ridge Regression

In [14]:
#create ridge pipeline
ridge_model = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0)) #alpha 1 is default, we can tune it if necessary
])

In [15]:
#train the model
ridge_model.fit(xn_train, yn_train)

Pipeline(steps=[('scaler', StandardScaler()), ('ridge', Ridge())])

In [16]:
#evaluate
y_pred = ridge_model.predict(xn_test)

print("R2 Score:", r2_score(yn_test, y_pred))
print("MSE:", mean_squared_error(yn_test, y_pred))

R2 Score: 0.9496571841994133
MSE: 30.024906280966118


In [17]:
#feature importance
#get coefficients
coefficients = ridge_model.named_steps['ridge'].coef_

#take absolute value (importance magnitude)
importance = np.abs(coefficients)

#create dataframe
ridge_importance = pandas.DataFrame({
    'feature': xn_train.columns,
    'importance': importance
})

#sort by importance
ridge_importance = ridge_importance.sort_values(by='importance', ascending=False)

ridge_importance

,feature,importance
6,task_completion_rate,20.037137
9,burnout_risk_High,2.905179
10,burnout_risk_Low,1.836760
11,burnout_risk_Medium,1.120967
1,screen_time_hours,0.554634
0,work_hours,0.527017
4,after_hours_work,0.254758
5,sleep_hours,0.239313
3,breaks_taken,0.092528
2,meetings_count,0.025448


In [18]:
#convert to ranking
ridge_importance['rank'] = ridge_importance['importance'].rank(ascending=False)

ridge_importance

,feature,importance,rank
6,task_completion_rate,20.037137,1.0
9,burnout_risk_High,2.905179,2.0
10,burnout_risk_Low,1.836760,3.0
11,burnout_risk_Medium,1.120967,4.0
1,screen_time_hours,0.554634,5.0
0,work_hours,0.527017,6.0
4,after_hours_work,0.254758,7.0
5,sleep_hours,0.239313,8.0
3,breaks_taken,0.092528,9.0
2,meetings_count,0.025448,10.0


## Decision Tree

### Regression Tree

In [19]:
tree = DecisionTreeRegressor()
parameters_tree = {# 'criterion':('squared_error','friedman_mse','absolute_error','poisson'),
                   'max_depth':[5,10,15,20],
                   'min_samples_split':[5,10,15,20],
                   'max_leaf_nodes':[5,10,20,50,None]
                  }
clf = GridSearchCV(tree, parameters_tree)
clf.fit(xn_train, yn_train)

GridSearchCV(estimator=DecisionTreeRegressor(),
             param_grid={'max_depth': [5, 10, 15, 20],
                         'max_leaf_nodes': [5, 10, 20, 50, None],
                         'min_samples_split': [5, 10, 15, 20]})

In [21]:
y_predicted_rt = pandas.DataFrame(clf.predict(xn_test))

vals = {}
vals['model'] = 'regression tree'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_predicted_rt, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

# feature importance ranking
tree_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': clf.best_estimator_.feature_importances_}).sort_values(by='importance', ascending=False)

tree_importance

,feature,importance
6,task_completion_rate,0.887926
10,burnout_risk_Low,0.101176
11,burnout_risk_Medium,0.009458
1,screen_time_hours,0.000975
0,work_hours,0.000465
2,meetings_count,0.000000
3,breaks_taken,0.000000
4,after_hours_work,0.000000
5,sleep_hours,0.000000
7,day_type_Weekday,0.000000


In [24]:
spearman_table

,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,0.035328,0.036523,0.017020,-0.094902,0.034909,0.015416,-0.088199,0.052561,-0.052561,0.013914,-0.058633,0.056210
0,regression tree,0.038433,0.037895,0.025671,-0.091210,0.028805,0.007181,-0.094695,0.050559,-0.050559,0.011876,-0.056988,0.055246
0,regression tree,0.038433,0.037895,0.025671,-0.091210,0.028805,0.007181,-0.094695,0.050559,-0.050559,0.011876,-0.056988,0.055246


In [25]:
x_nominal.columns

Index(['work_hours', 'screen_time_hours', 'meetings_count', 'breaks_taken',
       'after_hours_work', 'sleep_hours', 'task_completion_rate',
       'day_type_Weekday', 'day_type_Weekend', 'burnout_risk_High',
       'burnout_risk_Low', 'burnout_risk_Medium'],
      dtype='object')

### Random Forest

In [27]:
rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_depth=None,
    random_state=42
)
rf_reg.fit(xn_train, yn_train)
y_pred_rf = pandas.DataFrame(rf_reg.predict(xn_test))

vals = {}
vals['model'] = 'random forest'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_rf, method='spearman')
    
spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

# feature importance ranking
rf_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': rf_reg.feature_importances_}).sort_values(by='importance', ascending=False)

rf_importance

,feature,importance
6,task_completion_rate,0.871446
10,burnout_risk_Low,0.073866
5,sleep_hours,0.011070
1,screen_time_hours,0.010810
0,work_hours,0.008804
11,burnout_risk_Medium,0.007655
9,burnout_risk_High,0.006456
2,meetings_count,0.004305
3,breaks_taken,0.003580
4,after_hours_work,0.001449


## Boosting

### Gradient Boosting

In [28]:
gb_reg = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=3,
    random_state=42
)
gb_reg.fit(xn_train, yn_train)
y_pred_gb = pandas.DataFrame(gb_reg.predict(xn_test))

vals = {}
vals['model'] = 'gradient boosting'

for i in x_nominal.columns:
    vals[i] = x_nominal[i].corr(y_pred_gb, method='spearman')

spearman_table = pandas.concat([spearman_table, pandas.DataFrame([vals])])

# feature importance ranking
gb_importance = pandas.DataFrame({'feature': xn_train.columns,'importance': gb_reg.feature_importances_}).sort_values(by='importance', ascending=False)

gb_importance

,feature,importance
6,task_completion_rate,0.906876
10,burnout_risk_Low,0.067292
11,burnout_risk_Medium,0.010636
1,screen_time_hours,0.004572
9,burnout_risk_High,0.004503
0,work_hours,0.002676
5,sleep_hours,0.002620
2,meetings_count,0.000413
3,breaks_taken,0.000177
4,after_hours_work,0.000130


In [29]:
spearman_table


,model,work_hours,screen_time_hours,meetings_count,breaks_taken,after_hours_work,sleep_hours,task_completion_rate,day_type_Weekday,day_type_Weekend,burnout_risk_High,burnout_risk_Low,burnout_risk_Medium
0,linear regression,0.035328,0.036523,0.017020,-0.094902,0.034909,0.015416,-0.088199,0.052561,-0.052561,0.013914,-0.058633,0.056210
0,regression tree,0.038433,0.037895,0.025671,-0.091210,0.028805,0.007181,-0.094695,0.050559,-0.050559,0.011876,-0.056988,0.055246
0,regression tree,0.038433,0.037895,0.025671,-0.091210,0.028805,0.007181,-0.094695,0.050559,-0.050559,0.011876,-0.056988,0.055246
0,random forest,0.034162,0.034980,0.021563,-0.090377,0.031778,0.012042,-0.097186,0.054169,-0.054169,0.013809,-0.063711,0.061565
0,random forest,0.034162,0.034980,0.021563,-0.090377,0.031778,0.012042,-0.097186,0.054169,-0.054169,0.013809,-0.063711,0.061565
0,gradient boosting,0.031721,0.033298,0.019639,-0.090968,0.028420,0.009550,-0.095668,0.048071,-0.048071,0.011412,-0.062154,0.060828


### XGBoost

In [30]:
#create the model
xgb_model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)
# these search parameters are default/safe and can be altered

In [31]:
#train the model
xgb_model.fit(xn_train, yn_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)

In [32]:
#evaluate performance
y_pred_xgb = xgb_model.predict(xn_test)

print("R2 Score:", r2_score(yn_test, y_pred_xgb))
print("MSE:", mean_squared_error(yn_test, y_pred_xgb))

R2 Score: 0.9488235207683665
MSE: 30.522110618645502


In [33]:
#XGBoost handles feature importance in different ways. For research we use "gain" which refers to the average gain of splits.
importance_dict = xgb_model.get_booster().get_score(importance_type='gain')

In [34]:
#create dataframe with all features
xgb_importance = pandas.DataFrame({
    'feature': xn_train.columns
})

#map gain scores
xgb_importance['importance'] = xgb_importance['feature'].map(importance_dict)

#replace missing values (unused features) with 0
xgb_importance['importance'] = xgb_importance['importance'].fillna(0)

#sort
xgb_importance = xgb_importance.sort_values(by='importance', ascending=False)

xgb_importance

,feature,importance
10,burnout_risk_Low,12305.310547
6,task_completion_rate,8098.017578
9,burnout_risk_High,394.620026
1,screen_time_hours,77.440178
5,sleep_hours,66.025513
0,work_hours,55.726711
2,meetings_count,50.854389
4,after_hours_work,49.707619
3,breaks_taken,40.705776
7,day_type_Weekday,20.906706


In [35]:
#convert to spearman ranking
xgb_importance['rank'] = xgb_importance['importance'].rank(ascending=False)

xgb_importance

,feature,importance,rank
10,burnout_risk_Low,12305.310547,1.0
6,task_completion_rate,8098.017578,2.0
9,burnout_risk_High,394.620026,3.0
1,screen_time_hours,77.440178,4.0
5,sleep_hours,66.025513,5.0
0,work_hours,55.726711,6.0
2,meetings_count,50.854389,7.0
4,after_hours_work,49.707619,8.0
3,breaks_taken,40.705776,9.0
7,day_type_Weekday,20.906706,10.0
